In [ ]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
from UCI.pre_processing import  load_UCI_dataset, create_dataloaders

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 8640
Validation recordings: 960
Test recordings: 2400


100%|██████████| 8640/8640 [01:37<00:00, 88.55it/s] 


Skipped recordings: 4


100%|██████████| 960/960 [00:11<00:00, 86.56it/s] 


Skipped recordings: 0


100%|██████████| 2400/2400 [00:28<00:00, 85.48it/s] 


Skipped recordings: 1


In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

X_train: torch.Size([468372, 1, 1000])
y_train: torch.Size([468372, 2])
X_val: torch.Size([52933, 1, 1000])
y_val: torch.Size([52933, 2])
X_test: torch.Size([133841, 1, 1000])
y_test: torch.Size([133841, 2])


In [ ]:
print(X_train.dtype)
print(y_train.dtype)

print(torch.isnan(X_train).any())
print(torch.isnan(y_train).any())

torch.float32
torch.float32
tensor(False)
tensor(False)


In [ ]:
cat ConvTran/Models/model.py

import numpy as np
from torch import nn
from Models.AbsolutePositionalEncoding import tAPE, AbsolutePositionalEncoding, LearnablePositionalEncoding
from Models.Attention import Attention, Attention_Rel_Scl, Attention_Rel_Vec


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class Permute(nn.Module):
    def forward(self, x):
        return x.permute(1, 0, 2)


def model_factory(config):
    if config['Net_Type'][0] == 'T':
        model = Transformer(config, output_size=config['output_size'])
    elif config['Net_Type'][0] == 'CC-T':
        model = CasualConvTran(config, output_size=config['output_size'])
    else:
        model = ConvTran(config, output_size=config['output_size'])
    return model


class Transformer(nn.Module):
    def __init__(self, config, output_size):
        super().__init__()
        # Parameters Initialization -----------------------------------------------
        channel_size, seq_len = config['Data_sh

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

X: torch.Size([64, 1, 1000]) torch.float32
y: torch.Size([64, 2]) torch.float32


In [ ]:
config = {
    # Input
    'Data_shape': (32, 1, 1000),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [ ]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4215.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1101.)
  return F.conv2d(


Input : torch.Size([64, 1, 1000])
Output: torch.Size([64, 2])


In [ ]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

Prediction shape: torch.Size([64, 2])
Target shape: torch.Size([64, 2])
Loss: 97.55856323242188
Backward pass successful


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

Optimizer step successful


In [ ]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

Loss: 99.75161743164062
Training step successful


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [ ]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:

from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [ ]:
import os

config['epochs'] = 5
config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [ ]:
metrics = trainer.train_epoch(1)

print(metrics)

OrderedDict([('epoch', 1), ('loss', 81.72551026013262)])


In [ ]:
val_metrics, results = val_evaluator.evaluate(1)

print(val_metrics)

OrderedDict([('epoch', 1), ('loss', 61.84759914895525), ('SBP_MAE', np.float32(92.47198)), ('DBP_MAE', np.float32(32.223225)), ('SBP_RMSE', np.float32(95.13291)), ('DBP_RMSE', np.float32(34.03561))])


In [ ]:
config['epochs'] = 50

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/50 [00:00<?, ?it/s]2026-09-12 16:59:09,251 | INFO : Validation Summary: epoch: 1.000000 | loss: 10.354651 | SBP_MAE: 14.657890 | DBP_MAE: 7.026371 | SBP_RMSE: 19.223333 | DBP_RMSE: 10.044921 | 
2026-09-12 16:59:09,399 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 9.957665 | 
Training Epoch:   2%|▏         | 1/50 [13:28<10:59:53, 808.04s/it]


Best validation loss: 10.354651370090052
Saving best model for epoch: 1



2026-09-12 17:12:35,834 | INFO : Validation Summary: epoch: 2.000000 | loss: 10.308291 | SBP_MAE: 14.484869 | DBP_MAE: 7.106310 | SBP_RMSE: 18.982487 | DBP_RMSE: 9.766770 | 
2026-09-12 17:12:35,979 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 9.791229 | 
Training Epoch:   4%|▍         | 2/50 [26:54<10:45:44, 807.18s/it]


Best validation loss: 10.308290504610653
Saving best model for epoch: 2



2026-09-12 17:26:02,832 | INFO : Validation Summary: epoch: 3.000000 | loss: 10.288649 | SBP_MAE: 14.541194 | DBP_MAE: 7.011126 | SBP_RMSE: 19.242338 | DBP_RMSE: 9.855742 | 
2026-09-12 17:26:02,946 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 9.657876 | 
Training Epoch:   6%|▌         | 3/50 [40:21<10:32:12, 807.08s/it]


Best validation loss: 10.288648917142492
Saving best model for epoch: 3



2026-09-12 17:39:30,103 | INFO : Validation Summary: epoch: 4.000000 | loss: 10.313507 | SBP_MAE: 14.549254 | DBP_MAE: 7.052711 | SBP_RMSE: 19.212965 | DBP_RMSE: 9.817590 | 
2026-09-12 17:39:30,108 | INFO : Epoch 4 Training Summary: epoch: 4.000000 | loss: 9.559313 | 
Training Epoch:   8%|▊         | 4/50 [53:48<10:18:47, 807.11s/it]

2026-09-12 17:52:57,298 | INFO : Validation Summary: epoch: 5.000000 | loss: 10.090058 | SBP_MAE: 14.240875 | DBP_MAE: 6.913594 | SBP_RMSE: 18.806211 | DBP_RMSE: 9.719038 | 
2026-09-12 17:52:57,425 | INFO : Epoch 5 Training Summary: epoch: 5.000000 | loss: 9.468009 | 
Training Epoch:  10%|█         | 5/50 [1:07:16<10:05:23, 807.19s/it]


Best validation loss: 10.090057752945551
Saving best model for epoch: 5

